# Data privacy  

In [0]:
%sql
-- Data control on rows

CREATE FUNCTION columnf_filter(region STRING)
RETURN IF(is_member('admin'), true, region='US')

ALTER TABLE sales SET ROW FILTER columnf_filter ON region;

In [0]:
%sql
-- data control on columns
CREATE FUNCTION pii_mask(pii_column STRING)
RETURN CASE WHEN is_member('admin') THEN pii_column ELSE mask(pii_column) END;

ALTER TABLE sales ALTER COLUMN name SET MASK pii_mask;

# Audit Your Data

In [0]:
%sql
-- What tables are in the sales catalog ?
SELECT table_name
 FROM system.information_schema.tables 
 WHERE table_catalog = 'sales';

In [0]:
%sql
-- Who has access to this table ?
SELECT grantee, table_name, privilege_type
 FROM system.information_schema.table_privileges
 WHERE table_name = 'login_data_silver';

In [0]:
%sql
-- who last updated the gold tables and when ?
SELECT table_name, last_altered_by, last_altered 
FROM `system`.information_schema.tables
WHERE table_schema = 'table_gold'
ORDER BY 1,3 DESC;

In [0]:
%sql
-- Who owns this gold table ?
SELECT table_owner
FROM `system`.information_schema.tables
WHERE table_schema = 'bronze' AND table_name 'table_bronze';

# Audit logs

In [0]:
%sql
-- Who accesses this table the most ?
SELECT user_identity.email, count(*)
FROM system.access.audit
WHERE request_params.table_full_name = 'main.uc_deep_dive.login_data_silver'
AND service_name = 'unityCatalog'
AND action_name = 'generateTemporaryTableCredential'
GROUP BY 1
ORDER BY 2
DESC LIMIT 1;

In [0]:
%sql
-- who deleted this table ?
SELECT user
FROM system.access.audit
WHERE request_params.table_full_name = 'main.uc_deep_dive.login_data_silver'
AND action_name = 'deleteTable' AND service_name = 'unityCatalog';

In [0]:
%sql
-- what has this user accessed in the last 24h ?
SELECT request_params.table_full_name
FROM system.access.audit
WHERE user_identity.email = '<EMAIL>'
AND service_name = 'unityCatalog' AND action_name = 'generateTemporaryTableCredential'
AND datediff(now(), event_time) < 1;

In [0]:
%sql
-- what tables does this user access most frequently ?
SELECT request_prarms.table_full_name, count(*)
FROM system.access.audit
WHERE user_identity.email = '<EMAIL>'
AND service_name = 'unityCatalog'
AND action_name = 'generateTemporaryTableCredential'
GROUP BY 1
ORDER BY 2
DESC LIMIT 1;

# Lineage data

In [0]:
%sql
-- what tables are sourced from this table ?
SELECT target_table_full_name
FROM system.access.table_lineage
WHERE source_table_name = 'login_data_bronze';

In [0]:
%sql
-- what user queries read from this table ?
SELECT DISTINCT entity_type, entity_id, source_table_full_name
FROM system.access.table_lineage
WHERE source_table_name = 'login_data_silver';

# PII Security

In [0]:
# Anonimization PII data
from delta.tables import *
from spark.sql import functions as F
salt = 'BEAN'

def salted_hash(id):
    return F.sha2(F.concat(id, F.lit(salt)), 256)

@dlt.table
def user_lookup_hashed():
    return (
        dlt.read_stream("user_lookup")
        .select(
            salted_hash(F.col("id")).alias("id"),
            F.col("name"),
            F.col("email"),
        )
    )
# Pseudonymization strategy

@dlt.table
def registered_users_tokens():
    return (
        dlt.read_stream("registered_users")
        .select('user_id')
        .distinct()
        .withColumn("token", F.expr("uuid()"))
    )

@dlt.table
def user_lookup_tokinezed():
    return (
        dlt.read_stream('registered_users')
        .join(dlt.read('registered_users_tokens'), on='user_id', how='left')
        .drop('user_id')
        .withColumnRenamed('token', 'alt_id')
    )

# date of birth anonimization

def age_bins(dob_col): 
    age_col = F.floor(F.months_between(F.current_date(), dob_col) / 12).alias('age')
    return (
        F.when(age_col < 18, 'under 18')
        .when((age_col >= 18) & (age_col < 25, '18-24'))
        .when((age_col >= 25) & (age_col < 35, '25-34'))
        .when((age_col >= 35) & (age_col < 45, '35-44'))
        .when((age_col >= 45) & (age_col < 55, '45-54'))
        .when((age_col >= 55) & (age_col < 65, '55-64'))
        .when((age_col >= 65) & (age_col < 75, '65-74'))
        .when((age_col >= 75) & (age_col < 85, '75-84'))
        .when((age_col >= 85) & (age_col < 95, '85-94'))
        .when(age_col >= 95, '95+')
        .otherwise('invalid_age')
        .alias('age')
    )

@dlt.table
def user_age_bins():
    return (
        dlt.read_stream('user_lookup_hashed')
        .select('user_id', age_bins(F.col('dob')), 'gender', 'city', 'state')
    )    